# Notebook 08 — Ensemble Learning

Combine ALL FOUR base models (Logistic Regression, Random Forest, XGBoost, LightGBM) using a Soft Voting Ensemble (`voting='soft'`) to produce ONE FINAL prediction.


In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.ensemble import EasyEnsembleClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


In [2]:
# Load SMOTE-balanced training data and untouched test data
X_train = joblib.load("data/processed/X_train_smote.pkl")
y_train = joblib.load("data/processed/y_train_smote.pkl")

test_df = pd.read_csv("data/processed/test_dataset.csv")
X_test = test_df.drop("Maintenance_Required", axis=1)
y_test = test_df["Maintenance_Required"]

print("SMOTE Training shape:", X_train.shape, y_train.shape)
print("Untouched Test shape:", X_test.shape, y_test.shape)


SMOTE Training shape: (270496, 20) (270496,)
Untouched Test shape: (50000, 20) (50000,)


In [3]:
# Load pre-trained Base Models from Notebook 07
lr = joblib.load("models/logistic_regression_smote.pkl")
rf = joblib.load("models/random_forest_smote.pkl")
xgb = joblib.load("models/xgboost_smote.pkl")
lgbm = joblib.load("models/lightgbm_smote.pkl")

print("All 4 base models loaded successfully:")
print(" - Logistic Regression")
print(" - Random Forest")
print(" - XGBoost")
print(" - LightGBM")


All 4 base models loaded successfully:
 - Logistic Regression
 - Random Forest
 - XGBoost
 - LightGBM


In [4]:
# Build and train Soft Voting Ensemble using ALL FOUR models
print("=" * 60)
print("Training Soft Voting Ensemble (LR + RF + XGB + LightGBM)...")
print("=" * 60)

soft_voting_ensemble = VotingClassifier(
    estimators=[
        ('logistic_regression', lr),
        ('random_forest', rf),
        ('xgboost', xgb),
        ('lightgbm', lgbm)
    ],
    voting='soft'
)

soft_voting_ensemble.fit(X_train, y_train)
print("Soft Voting Ensemble trained successfully.")


Training Soft Voting Ensemble (LR + RF + XGB + LightGBM)...


E:\Predictive-vehicle-maintenance-with-road-analysis\Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Soft Voting Ensemble trained successfully.


In [5]:
# Evaluate Soft Voting Ensemble on untouched test set
voting_pred = soft_voting_ensemble.predict(X_test)
voting_prob = soft_voting_ensemble.predict_proba(X_test)[:, 1]

voting_results = {
    "Model": "Soft Voting Ensemble",
    "Accuracy": accuracy_score(y_test, voting_pred),
    "Precision": precision_score(y_test, voting_pred),
    "Recall": recall_score(y_test, voting_pred),
    "F1 Score": f1_score(y_test, voting_pred),
    "ROC-AUC": roc_auc_score(y_test, voting_prob)
}

voting_results_df = pd.DataFrame([voting_results])
print("Soft Voting Ensemble Evaluation:")
voting_results_df


Soft Voting Ensemble Evaluation:


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Soft Voting Ensemble,0.94036,0.93938,0.872066,0.904472,0.924784


In [6]:
# Train Stacking Ensemble for comparison
print("=" * 60)
print("Training Stacking Ensemble for architectural comparison...")
print("=" * 60)

stack_sample_idx = np.random.RandomState(42).choice(len(X_train), size=min(25000, len(X_train)), replace=False)
X_train_stack = X_train.iloc[stack_sample_idx]
y_train_stack = y_train.iloc[stack_sample_idx]

stack_model = StackingClassifier(
    estimators=[
        ('logistic_regression', LogisticRegression(max_iter=1000, random_state=42)),
        ('random_forest', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)),
        ('xgboost', XGBClassifier(n_estimators=50, random_state=42, eval_metric="logloss", n_jobs=-1)),
        ('lightgbm', LGBMClassifier(n_estimators=50, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=-1))
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=2,
    n_jobs=-1
)
stack_model.fit(X_train_stack, y_train_stack)

stack_pred = stack_model.predict(X_test)
stack_prob = stack_model.predict_proba(X_test)[:, 1]

stack_results_df = pd.DataFrame([{
    "Model": "Stacking Ensemble",
    "Accuracy": accuracy_score(y_test, stack_pred),
    "Precision": precision_score(y_test, stack_pred),
    "Recall": recall_score(y_test, stack_pred),
    "F1 Score": f1_score(y_test, stack_pred),
    "ROC-AUC": roc_auc_score(y_test, stack_prob)
}])


Training Stacking Ensemble for architectural comparison...


In [7]:
# Train Easy Ensemble for comparison
print("=" * 60)
print("Training Easy Ensemble for architectural comparison...")
print("=" * 60)

train_df = pd.read_csv("data/processed/train_dataset.csv")
X_train_orig = train_df.drop("Maintenance_Required", axis=1)
y_train_orig = train_df["Maintenance_Required"]

easy_ensemble = EasyEnsembleClassifier(n_estimators=10, random_state=42, n_jobs=-1)
easy_ensemble.fit(X_train_orig, y_train_orig)

easy_pred = easy_ensemble.predict(X_test)
easy_prob = easy_ensemble.predict_proba(X_test)[:, 1]

easy_results_df = pd.DataFrame([{
    "Model": "Easy Ensemble",
    "Accuracy": accuracy_score(y_test, easy_pred),
    "Precision": precision_score(y_test, easy_pred),
    "Recall": recall_score(y_test, easy_pred),
    "F1 Score": f1_score(y_test, easy_pred),
    "ROC-AUC": roc_auc_score(y_test, easy_prob)
}])


Training Easy Ensemble for architectural comparison...


In [8]:
# Save all ensemble models
os.makedirs("models", exist_ok=True)

joblib.dump(soft_voting_ensemble, "models/voting_ensemble_model.pkl")
joblib.dump(soft_voting_ensemble, "models/best_model_smote.pkl")
joblib.dump(stack_model, "models/final_stacking_model.pkl")
joblib.dump(easy_ensemble, "models/easy_ensemble_model.pkl")

print("All ensemble model artifacts saved successfully:")
print(" - models/voting_ensemble_model.pkl (PRIMARY FINAL ARCHITECTURE)")
print(" - models/best_model_smote.pkl")
print(" - models/final_stacking_model.pkl")
print(" - models/easy_ensemble_model.pkl")


All ensemble model artifacts saved successfully:
 - models/voting_ensemble_model.pkl (PRIMARY FINAL ARCHITECTURE)
 - models/best_model_smote.pkl
 - models/final_stacking_model.pkl
 - models/easy_ensemble_model.pkl
